## I want to see if games with certain words in the title sell better in some regions vs. others


In [18]:
import pandas as pd
import altair as alt
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import NMF



In [52]:
df = pd.read_csv('../data/cleaned_data/vgchartz-2024-cleaned.csv')
df.head(2)

,img,title,console,genre,publisher,developer,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update,has_critic_score,year
0,/games/boxart/full_6510540AmericaFrontccc.jpg,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,9.4,20.32,6.37,0.99,9.85,3.12,2013-09-17,NaN,1,2013.0
1,/games/boxart/full_5563178AmericaFrontccc.jpg,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,9.7,19.39,6.06,0.60,9.71,3.02,2014-11-18,2018-01-03,1,2014.0


### Vectorize

In [ ]:
# word_list = []

# for index, row in df.iterrows():
#     words = row['title'].split(' ')
#     print(words + word_list)
#     print(index)
#     break

vec = CountVectorizer(stop_words='english', binary=True)

X = vec.fit_transform(df['title'])



In [17]:
X.toarray()[5]

array([0, 0, 0, ..., 0, 0, 0], shape=(21341,))

### NMF to extract certain topics

In [ ]:
nmf = NMF(n_components=5, random_state=42)

W = nmf.fit_transform(X)
T = nmf.components_


In [31]:
W[0]

array([0.00023052, 0.00176645, 0.0002917 , 0.00225974, 0.0014219 ])

In [32]:
words = vec.get_feature_names_out()
print(nmf.components_.shape)
for i, topic in enumerate(nmf.components_):
    top_indexes = topic.argsort()[-5:]
    print(top_indexes)
    print(f"topic {i} contains {[words[x] for x in top_indexes]}")


(5, 21341)
[ 6210 20315  5392  5073  8676]
topic 0 contains ['final', 'war', 'edition', 'dragon', 'ii']
[ 4062 19220 17377  3123 20680]
topic 1 contains ['cup', 'tour', 'soccer', 'championship', 'world']
[ 5198    76 17104 20194 16518]
topic 2 contains ['ds', '1500', 'simple', 'vol', 'series']
[10777 15585 20376 17830 18190]
topic 3 contains ['lego', 'robot', 'wars', 'star', 'super']
[ 1208  2519 20111  5392  6814]
topic 4 contains ['arcade', 'boy', 'video', 'edition', 'game']


In [53]:
new_cols = ['topic_' + str(i) for i, top in enumerate(T)]
print(new_cols)

df[new_cols] = W

['topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4']


In [49]:
df.columns

Index(['img', 'title', 'console', 'genre', 'publisher', 'developer',
       'critic_score', 'total_sales', 'na_sales', 'jp_sales', 'pal_sales',
       'other_sales', 'release_date', 'last_update', 'has_critic_score',
       'year', 'topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4'],
      dtype='object')

In [71]:
weighted_regional_sales_per_topic = pd.DataFrame({
    'topic':  [top for top in new_cols],
    'na_sales_weighted': [(df['na_sales'] * df[top]).sum() for top in new_cols],
    'jp_sales_weighted': [(df['jp_sales'] * df[top]).sum() for top in new_cols],
    'pal_sales_weighted': [(df['pal_sales'] * df[top]).sum() for top in new_cols],
    'other_sales_weighted': [(df['other_sales'] * df[top]).sum() for top in new_cols],
    'total_sales_weighted': [(df['total_sales'] * df[top]).sum() for top in new_cols]
})

melted_sales_region = weighted_regional_sales_per_topic.melt(value_vars=['na_sales_weighted', 'pal_sales_weighted', 'jp_sales_weighted', 'other_sales_weighted'], value_name="sales", var_name="region", id_vars="topic", ignore_index=False)

melted_sales_region

,topic,region,sales
0,topic_0,na_sales_weighted,14.491331
1,topic_1,na_sales_weighted,17.794439
2,topic_2,na_sales_weighted,4.721478
3,topic_3,na_sales_weighted,29.625965
4,topic_4,na_sales_weighted,18.750453
0,topic_0,pal_sales_weighted,7.886397
1,topic_1,pal_sales_weighted,12.983334
2,topic_2,pal_sales_weighted,2.355307
3,topic_3,pal_sales_weighted,16.586272
4,topic_4,pal_sales_weighted,9.289707


In [75]:
region_sales = alt.Chart(melted_sales_region).mark_bar().encode(
    x='topic:N', 
    y='sales:Q',
    color='region:N'
).properties(title="Regional Sales Weighted by Topic")

In [80]:

rows = []
for index, topic in enumerate(nmf.components_):
    weights = nmf.components_[index]
    top_indexes = topic.argsort()[-20:]

    for i in top_indexes:
        rows.append({
            'topic': 'topic_' + str(index),
            'word': words[i],
            'weight': weights[i]
        })

topic_word_weights = pd.DataFrame(rows)


topic_word_weights.head(5)


,topic,word,weight
0,topic_0,force,0.113189
1,topic_0,black,0.115336
2,topic_0,hearts,0.119863
3,topic_0,assassin,0.122452
4,topic_0,heroes,0.126961


In [84]:
rng = np.random.default_rng(42)

topic_word_weights['x'] = rng.uniform(0,1,size=topic_word_weights.shape[0])
topic_word_weights['y'] = rng.uniform(0,1,size=topic_word_weights.shape[0])

In [85]:
topic_word_weights

,topic,word,weight,x,y
0,topic_0,force,0.113189,0.773956,0.908581
1,topic_0,black,0.115336,0.438878,0.699707
2,topic_0,hearts,0.119863,0.858598,0.265870
3,topic_0,assassin,0.122452,0.697368,0.969176
4,topic_0,heroes,0.126961,0.094177,0.778751
...,...,...,...,...,...
95,topic_4,arcade,0.293670,0.630283,0.179268
96,topic_4,boy,0.296927,0.361813,0.599383
97,topic_4,video,0.766419,0.087650,0.874562
98,topic_4,edition,1.011163,0.118006,0.196435


In [88]:


chart = (
    alt.Chart(topic_word_weights)
    .mark_text()
    .encode(
        x='x:Q',
        y='y:Q',
        text='word:N',
        size=alt.Size('weight:Q', scale=alt.Scale(range=[10, 60])),
        opacity=alt.Opacity('weight:Q', scale=alt.Scale(range=[0.4, 1])),
        color=alt.value('#4C78A8'),
        tooltip=['word', 'weight']
    )
    .properties(width=250, height=200)
    .facet(column='topic:N')
)

region_sales & chart

alt.VConcatChart(...)